# Import Modules

In [13]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from utils import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

In [14]:
# 3 Datasets
df_chd_framingham_heart_intersection = get_unscaled_combined("chd-framingham-heart-intersection")
df_chd_framingham_heart_union = get_unscaled_combined("chd-framingham-heart-union")
df_chd_framingham_stroke_intersection = get_unscaled_combined("chd-framingham-stroke-intersection")
df_chd_framingham_stroke_union = get_unscaled_combined("chd-framingham-stroke-union")

# 2 Datasets
df_chd_framingham_intersection = get_unscaled_combined("chd-framingham-intersection")
df_chd_framingham_union = get_unscaled_combined("chd-framingham-union")
df_chd_heart_intersection = get_unscaled_combined("chd-heart-intersection")
df_chd_heart_union = get_unscaled_combined("chd-heart-union")

In [15]:
union_datasets = {
    "chd-framingham-heart-union": df_chd_framingham_heart_union,
    "chd-framingham-stroke-union": df_chd_framingham_stroke_union,
    "chd-framingham-union": df_chd_framingham_union,
    "chd-heart-union": df_chd_heart_union
}

intersection_datasets = {
    "chd-framingham-heart-intersection": df_chd_framingham_heart_intersection,
    "chd-framingham-stroke-intersection": df_chd_framingham_stroke_intersection,
    "chd-framingham-intersection": df_chd_framingham_intersection,
    "chd-heart-intersection": df_chd_heart_intersection
}

In [16]:
def check_missing_values(df, dataset_name="Dataset"):
    print(f"=== Missing Value Report: {dataset_name} ===")
    
    # 1. Check if ANY missing values exist overall
    has_missing = df.isnull().values.any()

    if has_missing:
        # 2. Count missing values for each column
        missing_counts = df.isnull().sum()
        
        # Filter to show only columns that actually have missing data
        missing_columns = missing_counts[missing_counts > 0]
        
        # 3. Calculate the percentage of missing data per column
        missing_percentages = (missing_columns / len(df)) * 100
        
        # Combine counts and percentages into a clean table
        summary_df = pd.DataFrame({
            'Missing Count': missing_columns,
            'Percentage (%)': missing_percentages.round(2)
        }).sort_values(by='Missing Count', ascending=False)
        
        print("Columns with missing values:")
        print(summary_df)
        print("\n")
    else:
        print("Great news! This dataset is completely full with no missing values.\n")


check_missing_values(df_chd_framingham_union, "CHD-Framingham Union")
check_missing_values(df_chd_framingham_intersection, "CHD-Framingham Intersection")
check_missing_values(df_chd_heart_union, "CHD-Heart Union")
check_missing_values(df_chd_heart_intersection, "CHD-Heart Intersection")
check_missing_values(df_chd_framingham_heart_union, "CHD-Framingham-Heart Union")
check_missing_values(df_chd_framingham_heart_intersection, "CHD-Framingham-Heart Intersection")
check_missing_values(df_chd_framingham_stroke_union, "CHD-Framingham-Stroke Union")
check_missing_values(df_chd_framingham_stroke_intersection, "CHD-Framingham-Stroke Intersection")

=== Missing Value Report: CHD-Framingham Union ===
Columns with missing values:
                     Missing Count  Percentage (%)
tobacco                       4240           90.17
adiposity                     4240           90.17
famhist                       4240           90.17
typea                         4240           90.17
alcohol                       4240           90.17
ldl                           4240           90.17
hypertension_status            462            9.83
hr                             462            9.83
dbp                            462            9.83
chol                           462            9.83
diabetes_status                462            9.83
num_cigs_per_day               462            9.83
stroke_status                  462            9.83
bp_status                      462            9.83
smoking_status                 462            9.83
edu                            462            9.83
sex                            462            9.83
gl

# Data Preprocessing

In [17]:
def process_union(df, name, drop_threshold=0.75):
    df_processed = df.copy()
    
    # 1. DROP columns with extreme missing values (e.g., > 75% missing)
    missing_fractions = df_processed.isnull().mean()
    cols_to_drop = missing_fractions[missing_fractions > drop_threshold].index
    
    if len(cols_to_drop) > 0:
        print(f"⚠️ [{name}] Dropping columns (> {drop_threshold*100}% missing): {list(cols_to_drop)}")
        df_processed.drop(columns=cols_to_drop, inplace=True)

    numeric_cols = df_processed.select_dtypes(include=['number']).columns
    categorical_cols = df_processed.select_dtypes(exclude=['number']).columns

    # 2. IMPUTE the remaining columns
    if len(numeric_cols) > 0:
        num_imputer = SimpleImputer(strategy='most_frequent')
        df_processed[numeric_cols] = num_imputer.fit_transform(df_processed[numeric_cols])

    if len(categorical_cols) > 0:
        cat_imputer = SimpleImputer(strategy='most_frequent')
        df_processed[categorical_cols] = cat_imputer.fit_transform(df_processed[categorical_cols])

    # 3. SCALE the numeric features
    scaler = MinMaxScaler()
    if len(numeric_cols) > 0:
        df_processed[numeric_cols] = scaler.fit_transform(df_processed[numeric_cols])

    set_preprocessed_combined(df_processed, scaler, name)
    print(f"✅ Processed Union Dataset: {name}\n")

print("--- Processing Union Datasets ---")
for name, df in union_datasets.items():
    process_union(df, name)

--- Processing Union Datasets ---
⚠️ [chd-framingham-heart-union] Dropping columns (> 75.0% missing): ['tobacco', 'ldl', 'adiposity', 'famhist', 'typea', 'alcohol', 'cp_type', 'fbs', 'ecg', 'angina', 'oldpeak', 'slope', 'mv', 'thal']
✅ Processed Union Dataset: chd-framingham-heart-union

⚠️ [chd-framingham-stroke-union] Dropping columns (> 75.0% missing): ['tobacco', 'ldl', 'adiposity', 'famhist', 'typea', 'alcohol']
✅ Processed Union Dataset: chd-framingham-stroke-union

⚠️ [chd-framingham-union] Dropping columns (> 75.0% missing): ['tobacco', 'ldl', 'adiposity', 'famhist', 'typea', 'alcohol']
✅ Processed Union Dataset: chd-framingham-union

✅ Processed Union Dataset: chd-heart-union



In [18]:
def process_intersection(df, name):
    df_processed = df.copy()
    numeric_cols = df_processed.select_dtypes(include=['number']).columns

    scaler = MinMaxScaler()
    if len(numeric_cols) > 0:
        df_processed[numeric_cols] = scaler.fit_transform(df_processed[numeric_cols])

    set_preprocessed_combined(df_processed, scaler, name)
    print(f"✅ Processed Intersection Dataset: {name}")
    
print("\n--- Processing Intersection Datasets ---")
for name, df in intersection_datasets.items():
    process_intersection(df, name)


--- Processing Intersection Datasets ---
✅ Processed Intersection Dataset: chd-framingham-heart-intersection
✅ Processed Intersection Dataset: chd-framingham-stroke-intersection
✅ Processed Intersection Dataset: chd-framingham-intersection
✅ Processed Intersection Dataset: chd-heart-intersection
